# Cancer Predictive Modeling


## Install and Import Libraries

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
from sklearn.decomposition import PCA

sns.set(style="whitegrid")
np.random.seed(42)

## Upload Dataset

In [ ]:
from zipfile import ZipFile

with ZipFile("/content/archive (2).zip", 'r') as zip_ref:
    zip_ref.extractall("/content/cancer_data")

import os
os.listdir("/content/cancer_data")


In [ ]:
import pandas as pd

df = pd.read_csv("/content/cancer_data/The_Cancer_data_1500_V2.csv")
df.head()


In [ ]:
numeric_features = ['Age', 'BMI', 'PhysicalActivity', 'AlcoholIntake']

plt.figure(figsize=(16, 4))
for i, feature in enumerate(numeric_features, 1):
    plt.subplot(1, len(numeric_features), i)
    sns.boxplot(x='Diagnosis', y=feature, data=df)
    plt.title(f"{feature} by Diagnosis")
plt.tight_layout()
plt.show()


## Split features and labels

In [ ]:
X = df.drop(columns=['Diagnosis'])
y = df['Diagnosis']


## Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## Standardize features

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

## Logistic Regression

In [ ]:
logreg = LogisticRegression(max_iter=2000, solver='lbfgs')
logreg.fit(X_train_s, y_train)

pred_lr = logreg.predict(X_test_s)
proba_lr = logreg.predict_proba(X_test_s)[:, 1]

print("Logistic Regression Accuracy:", round(accuracy_score(y_test, pred_lr), 3))
print(classification_report(y_test, pred_lr))


## Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest Accuracy:", round(accuracy_score(y_test, pred_rf), 3))
print(classification_report(y_test, pred_rf))


## Confusion Matrix (Random Forest)

In [ ]:
cm = confusion_matrix(y_test, pred_rf)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Random Forest)")
plt.show()

## PCA 2D Projection (Train Set)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train_s)

plt.figure(figsize=(6,5))
sns.scatterplot(
    x=X_pca[:,0], y=X_pca[:,1],
    hue=y_train,
    palette=['tab:blue','tab:orange'],
    alpha=0.8
)
plt.title("PCA 2D Projection (Train Set) — color=class")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Diagnosis")
plt.show()

## Decision Tree Model

In [ ]:
from sklearn.tree import DecisionTreeClassifier


dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)


y_pred_dt = dt_model.predict(X_test)


print("Decision Tree Results")
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))
print("Classification Report:\n", classification_report(y_test, y_pred_dt))



## ROC Curve (Random Forest)

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt


proba_rf = rf.predict_proba(X_test)[:,1]
proba_dt = dt_model.predict_proba(X_test)[:,1]


fpr_rf, tpr_rf, _ = roc_curve(y_test, proba_rf)
roc_auc_rf = roc_auc_score(y_test, proba_rf)

fpr_dt, tpr_dt, _ = roc_curve(y_test, proba_dt)
roc_auc_dt = roc_auc_score(y_test, proba_dt)


plt.figure(figsize=(7,6))
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_rf:.3f})', color='blue')
plt.plot(fpr_dt, tpr_dt, label=f'Decision Tree (AUC = {roc_auc_dt:.3f})', color='orange')
plt.plot([0,1], [0,1], 'k--')  # diagonal line
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()


## Conclusion


In this mini project, I used a cancer dataset to predict diagnosis based on patient features.
Three machine learning models : Logistic Regression, Decision Tree, and Random Forest were trained and evaluated.

All models achieved good accuracy, with Random Forest performing the best in terms of overall classification metrics and ROC AUC. The PCA 2D projection shows some separation between classes but highlights the complexity of the feature space.

This project demonstrates that machine learning can capture patterns in biomedical data, but simple 2D visualizations may not fully reflect these relationships.

It is important to note that the dataset, while realistic, is limited in size, and real-world clinical predictions would require larger datasets, biological validation, and careful interpretation.

Overall, the project illustrates a complete workflow of data preprocessing, scaling, model training, evaluation, and visualization for predictive modeling in a biomedical context.
